# MCP Integration [Step 4 - Model Context Protocol with CrewAI]

> **MLCourse - Agentic AI - CrewAI Flows and Orchestration**

The Model Context Protocol (MCP) is an open standard for connecting AI
agents to external tools and data sources. CrewAI integrates with MCP
servers via `MCPServerStdio`, `MCPServerSSE`, and `MCPServerHTTP` classes,
letting agents use tools exposed by MCP servers as if they were native
CrewAI tools.

## What you will learn

1. The three MCP transport types: Stdio, SSE, and HTTP.
2. Configuring `MCPServerStdio`, `MCPServerSSE`, `MCPServerHTTP`.
3. Connecting MCP servers to CrewAI Agents via the `mcp` parameter.
4. Using `@tool` decorator for custom tool creation.
5. Security considerations for MCP integration.

## Key takeaways

- MCP servers expose tools that agents can call during execution.
- Stdio transport runs a local process; SSE and HTTP connect to remote servers.
- The `mcp` parameter on `Agent` accepts a list of MCP server configs.
- All MCP code should be guarded with try/except for graceful degradation.

In [ ]:
# --- Standard library imports -------------------------------------------------
import os                           # Environment variable access
from pathlib import Path            # OOP path handling

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv      # Load .env into os.environ

TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root resolved to:", TRACK)

## 1 -- Verify Ollama availability

In [ ]:
from langchain_ollama import ChatOllama

try:
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    LLM_AVAILABLE = True
    print("[GREEN] Ollama reachable -- full pipeline will run")
except Exception as exc:
    LLM_AVAILABLE = False
    print("[WARN] Ollama not reachable:", exc)
    print("MCP integration patterns demonstrated without LLM calls")

## 2 -- Import CrewAI MCP classes

The MCP integration lives in `crewai.mcp`. The three transport types
correspond to different ways of communicating with MCP servers:

- **Stdio**: Runs a local process and communicates via stdin/stdout.
- **SSE**: Connects to a Server-Sent Events endpoint.
- **HTTP**: Connects to an HTTP endpoint (supports streaming).

In [ ]:
from crewai import Agent, Task, Crew
from crewai.tools import tool  # Decorator for creating custom tools

# Import MCP server configs -- guard for environments where MCP is not installed
try:
    from crewai.mcp import MCPServerStdio, MCPServerSSE, MCPServerHTTP
    MCP_AVAILABLE = True
    print("MCP server classes imported successfully")
except ImportError as exc:
    MCP_AVAILABLE = False
    print(f"MCP classes not available: {exc}")
    print("Showing API patterns only")

## 3 -- MCPServerStdio: Local process transport

`MCPServerStdio` launches a local process (e.g., `npx`, `python`, `node`)
and communicates with it via stdin/stdout. This is the most common
transport for local MCP servers.

Common use cases:
- File system access via `@modelcontextprotocol/server-filesystem`.
- Database queries via custom MCP servers.
- API integrations via community MCP servers.

In [ ]:
# Example: Filesystem MCP server (Stdio transport)
# This would connect to a local MCP server that provides file operations.
# Guarded with try/except since the actual MCP server may not be installed.

filesystem_server_config = None
if MCP_AVAILABLE:
    try:
        filesystem_server_config = MCPServerStdio(
            command="npx",                          # Command to run
            args=[                                  # Arguments for the command
                "-y",
                "@modelcontextprotocol/server-filesystem",
                str(TRACK),                         # Allow access to track root
            ],
            env=None,                               # Optional environment variables
            cache_tools_list=True,                  # Cache tool list for performance
        )
        print("Filesystem MCP server config created (Stdio transport)")
        print(f"  Command: npx -y @modelcontextprotocol/server-filesystem {TRACK}")
    except Exception as exc:
        print(f"Could not create Stdio config: {exc}")
else:
    print("MCP not available -- showing API pattern")
    print("MCPServerStdio(command='npx', args=['-y', '@modelcontextprotocol/server-filesystem', '/path'])")

## 4 -- MCPServerSSE: Server-Sent Events transport

`MCPServerSSE` connects to a remote MCP server via Server-Sent Events.
This is used when the MCP server is running as a web service.

Common use cases:
- Cloud-hosted MCP servers.
- Shared team MCP servers.
- Managed MCP services.

In [ ]:
# Example: Remote SSE MCP server
sse_server_config = None
if MCP_AVAILABLE:
    try:
        sse_server_config = MCPServerSSE(
            url="http://localhost:8080/sse",        # SSE endpoint URL
            headers={                               # Optional HTTP headers
                "Authorization": "Bearer your-token-here",
            },
            cache_tools_list=True,
        )
        print("SSE MCP server config created")
        print("  URL: http://localhost:8080/sse")
        print("  Note: This is an example URL -- the server must be running")
    except Exception as exc:
        print(f"Could not create SSE config: {exc}")
else:
    print("MCP not available -- showing API pattern")
    print("MCPServerSSE(url='http://localhost:8080/sse', headers={'Authorization': 'Bearer ...'})")

## 5 -- MCPServerHTTP: HTTP transport with streaming

`MCPServerHTTP` connects to an HTTP endpoint and supports both regular
HTTP and streaming responses. The `streamable` parameter controls whether
to use streaming.

Common use cases:
- REST API-based MCP servers.
- High-performance streaming integrations.
- Load-balanced MCP server clusters.

In [ ]:
# Example: HTTP MCP server
http_server_config = None
if MCP_AVAILABLE:
    try:
        http_server_config = MCPServerHTTP(
            url="http://localhost:3000/mcp",        # HTTP endpoint
            headers={                               # Optional headers
                "X-API-Key": "your-api-key",
            },
            streamable=True,                        # Enable streaming
            cache_tools_list=True,
        )
        print("HTTP MCP server config created")
        print("  URL: http://localhost:3000/mcp")
        print("  Streaming: enabled")
    except Exception as exc:
        print(f"Could not create HTTP config: {exc}")
else:
    print("MCP not available -- showing API pattern")
    print("MCPServerHTTP(url='http://localhost:3000/mcp', streamable=True)")

## 6 -- Connecting MCP servers to agents

MCP servers are connected to agents via the `mcp` parameter. An agent
can have multiple MCP servers, giving it access to tools from all of them.
The agent automatically discovers available tools from each server.

The `tool_filter` parameter on MCP servers lets you restrict which tools
from the server the agent can use -- useful for security and focus.

In [ ]:
# Example: Agent with MCP tools
# Guarded since we may not have actual MCP servers running

if MCP_AVAILABLE and filesystem_server_config:
    try:
        mcp_agent = Agent(
            role="File System Agent",
            goal="Access and manipulate files using MCP tools",
            backstory="You are an agent with access to the local file system via MCP.",
            llm=ChatOllama(model="llama3.1:8b", temperature=0) if LLM_AVAILABLE else None,
            mcp=[filesystem_server_config],  # List of MCP server configs
            verbose=False,
        )
        print("Agent with MCP tools created")
        print(f"  MCP servers: {len(mcp_agent.mcp)}")
    except Exception as exc:
        print(f"Could not create MCP agent: {exc}")
else:
    print("MCP or LLM not available -- showing API pattern")
    print("""
    mcp_agent = Agent(
        role="File System Agent",
        goal="Access files via MCP",
        backstory="...",
        llm=ChatOllama(model='llama3.1:8b'),
        mcp=[filesystem_server_config],  # List of MCP servers
    )
    """)

## 7 -- Tool filter for security

MCP servers can expose many tools. The `tool_filter` parameter lets you
restrict which tools an agent can use. You can pass a callable that
receives the tool name and returns True/False.

This is critical for security: you don't want an agent using destructive
tools (like file deletion) when it only needs read access.

In [ ]:
# Example: Tool filter that only allows read operations
def read_only_filter(tool_name: str) -> bool:
    """Only allow tools that start with 'read' or 'list'."""
    allowed_prefixes = ("read", "list", "get", "search")
    return any(tool_name.lower().startswith(prefix) for prefix in allowed_prefixes)


if MCP_AVAILABLE:
    try:
        filtered_server = MCPServerStdio(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(TRACK)],
            tool_filter=read_only_filter,  # Only read-only tools
            cache_tools_list=True,
        )
        print("Filtered MCP server config created")
        print("  Tool filter: only read/list/get/search tools allowed")
        print("  This prevents the agent from writing or deleting files")
    except Exception as exc:
        print(f"Could not create filtered config: {exc}")
else:
    print("MCP not available -- showing API pattern")
    print("""
    def read_only_filter(tool_name: str) -> bool:
        return tool_name.startswith(('read', 'list', 'get'))

    filtered_server = MCPServerStdio(
        command='npx',
        args=['-y', '@modelcontextprotocol/server-filesystem', '/path'],
        tool_filter=read_only_filter,
    )
    """)

## 8 -- Multiple MCP servers on one agent

An agent can connect to multiple MCP servers simultaneously. This is
powerful for agents that need tools from different sources -- e.g., a
research agent that needs both web search and database access.

In [ ]:
# Example: Agent with multiple MCP servers
if MCP_AVAILABLE and filesystem_server_config and sse_server_config:
    try:
        multi_mcp_agent = Agent(
            role="Multi-Tool Researcher",
            goal="Research using multiple data sources via MCP",
            backstory="You have access to files and remote APIs via MCP.",
            llm=ChatOllama(model="llama3.1:8b", temperature=0) if LLM_AVAILABLE else None,
            mcp=[
                filesystem_server_config,  # Local file access
                sse_server_config,         # Remote API access
            ],
            verbose=False,
        )
        print("Agent with multiple MCP servers created")
        print(f"  MCP servers: {len(multi_mcp_agent.mcp)}")
    except Exception as exc:
        print(f"Could not create multi-MCP agent: {exc}")
else:
    print("MCP servers not available -- showing API pattern")
    print("""
    multi_mcp_agent = Agent(
        role="Multi-Tool Researcher",
        goal="Research using multiple data sources",
        backstory="...",
        llm=ChatOllama(model='llama3.1:8b'),
        mcp=[
            filesystem_server_config,  # Local files
            sse_server_config,         # Remote API
            http_server_config,        # HTTP endpoint
        ],
    )
    """)

## 9 -- Custom tools with @tool decorator

While MCP provides external tool integration, you can also create custom
tools using the `@tool` decorator. These tools are defined in Python and
don't require an MCP server. They're useful for:

- Simple utility functions.
- Wrapping existing APIs.
- Quick prototypes before building a full MCP server.

In [ ]:
@tool("word_count")
def word_count(text: str) -> str:
    """Count the number of words, characters, and sentences in the given text.

    Args:
        text: The text to analyze.
    """
    words = len(text.split())
    chars = len(text)
    sentences = text.count(".") + text.count("!") + text.count("?")
    return f"Words: {words}, Characters: {chars}, Sentences: {sentences}"


@tool("sentiment_hint")
def sentiment_hint(text: str) -> str:
    """Provide a basic sentiment hint based on keyword presence.

    Args:
        text: The text to analyze for sentiment.
    """
    positive_words = ["good", "great", "excellent", "amazing", "wonderful"]
    negative_words = ["bad", "terrible", "awful", "horrible", "poor"]
    text_lower = text.lower()
    pos = sum(1 for w in positive_words if w in text_lower)
    neg = sum(1 for w in negative_words if w in text_lower)
    if pos > neg:
        return "Positive sentiment detected"
    elif neg > pos:
        return "Negative sentiment detected"
    return "Neutral sentiment detected"


# Test the custom tools
sample_text = "This is a great example of custom tool creation in CrewAI."
print(f"Text: {sample_text}")
print(f"Word count: {word_count(sample_text)}")
print(f"Sentiment:  {sentiment_hint(sample_text)}")

## 10 -- Using custom tools with agents

Custom tools are passed to agents via the `tools` parameter. The agent
can call them during execution alongside any MCP tools. This gives you
a unified interface for both local and external tools.

In [ ]:
# Create an agent with custom tools
custom_tool_agent = Agent(
    role="Text Analyzer",
    goal="Analyze text using custom tools for word count and sentiment",
    backstory="You are an expert at text analysis using specialized tools.",
    llm=ChatOllama(model="llama3.1:8b", temperature=0) if LLM_AVAILABLE else None,
    tools=[word_count, sentiment_hint],  # Custom tools
    verbose=False,
)

print("Agent with custom tools created")
print(f"  Tools: {[t.name for t in custom_tool_agent.tools]}")

## 11 -- Running a crew with MCP tools

When you run a Crew with an MCP-equipped agent, the agent automatically
discovers and uses tools from the configured MCP servers. The MCP
connection is established when the crew starts and torn down when it ends.

In [ ]:
# Example: Running a crew with custom tools (no MCP servers needed)
if LLM_AVAILABLE:
    analysis_task = Task(
        description="Analyze this text for word count and sentiment: {text}",
        expected_output="A report with word count and sentiment analysis.",
        agent=custom_tool_agent,
    )

    crew = Crew(
        agents=[custom_tool_agent],
        tasks=[analysis_task],
        verbose=False,
    )

    result = crew.kickoff(
        inputs={"text": "CrewAI makes it easy to build multi-agent systems with MCP integration."}
    )
    print(f"\nAnalysis result:\n{result}")
else:
    print("Ollama not available -- showing crew structure only")
    print("Crew would use custom tools: word_count, sentiment_hint")

## 12 -- Security considerations

When using MCP servers, keep these security practices in mind:

1. **Tool filtering**: Always use `tool_filter` to restrict which tools
   agents can access. Never give an agent more tools than it needs.
2. **Environment isolation**: MCP servers running via Stdio have access
   to the same filesystem as the agent. Use `env` parameter to control
   environment variables.
3. **Authentication**: For SSE and HTTP servers, always use proper
   authentication headers. Never hardcode tokens in source code.
4. **Sandboxing**: Run MCP servers in containers or sandboxes when
   handling untrusted data.
5. **Caching**: Use `cache_tools_list=True` to avoid repeated tool
   discovery, which reduces attack surface and improves performance.

In [ ]:
print("=" * 60)
print("SECURITY CHECKLIST FOR MCP INTEGRATION")
print("=" * 60)
print()
print("[OK] Use tool_filter to restrict agent tool access")
print("[OK] Store MCP server tokens in .env, never in code")
print("[OK] Use cache_tools_list=True for performance")
print("[OK] Run Stdio servers in sandboxed environments")
print("[OK] Validate MCP server URLs and endpoints")
print("[OK] Monitor agent tool usage in production")

## 13 -- Summary

MCP integration in CrewAI provides a standardized way to connect agents
to external tools and data sources. The three transport types cover
local, remote, and streaming use cases.

| Transport | Class | Use Case |
|-----------|-------|----------|
| Stdio | `MCPServerStdio` | Local process (npx, python, node) |
| SSE | `MCPServerSSE` | Remote Server-Sent Events |
| HTTP | `MCPServerHTTP` | REST API with optional streaming |

Key patterns:
- Pass MCP configs via `Agent(mcp=[...])`.
- Use `tool_filter` for security.
- Use `@tool` for custom Python tools.
- Always guard MCP code with try/except.

In [ ]:
print("=" * 60)
print("MODULE SUMMARY -- MCP Integration")
print("=" * 60)
print()
print("Transport types:")
print("  MCPServerStdio(command, args, env, tool_filter)")
print("  MCPServerSSE(url, headers, tool_filter)")
print("  MCPServerHTTP(url, headers, streamable, tool_filter)")
print()
print("Agent integration:")
print("  Agent(..., mcp=[server1, server2], tools=[custom_tool1])")
print()
print("Custom tools:")
print("  @tool('name')  -- decorator for Python functions")
print()
print("Security:")
print("  tool_filter  -- restrict which tools agents can use")
print("  env          -- control environment variables for Stdio servers")
print("  headers      -- authentication for SSE/HTTP servers")